# Laboratorio 6: análisis de redes sociales en YouTube

**Avance: ejercicios 1 a 4**  
**Integrantes:** Milton Polanco y Osman de León

En este notebook se trabajan la carga, integración, limpieza, análisis exploratorio y construcción de la red bipartita.

## 0. Configuración

Se definen rutas, estilo gráfico, palabras vacías y funciones auxiliares. Los identificadores permanecen como texto y nunca se sustituyen por nombres visibles.

In [ ]:
%matplotlib inline
from __future__ import annotations

import json
import re
import unicodedata
from collections import Counter
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
FIGURES = ROOT / "reporte" / "figuras"
TABLES = ROOT / "reporte" / "tablas"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "figure.dpi": 140,
        "savefig.dpi": 220,
    }
)
sns.set_theme(style="whitegrid", font="serif")


STOPWORDS = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "aquel",
    "aquella", "aquellas", "aquello", "aquellos", "aqui", "asi", "aun",
    "aunque", "bajo", "bien", "cada", "casi", "como", "con", "contra",
    "cual", "cuando", "de", "del", "desde", "donde", "dos", "durante",
    "e", "el", "ella", "ellas", "ello", "ellos", "en", "entre", "era",
    "eran", "es", "esa", "esas", "ese", "eso", "esos", "esta", "estan",
    "estar", "este", "esto", "estos", "fue", "fuera", "fueron", "ha",
    "hace", "hacia", "han", "hasta", "hay", "la", "las", "le", "les",
    "lo", "los", "mas", "me", "mi", "mis", "mismo", "mucho", "muy",
    "nada", "ni", "no", "nos", "nuestra", "nuestro", "o", "otra", "otro",
    "para", "pero", "poco", "por", "porque", "que", "quien", "se", "sea",
    "ser", "si", "sin", "sobre", "son", "su", "sus", "tambien", "te",
    "tiene", "todo", "tu", "un", "una", "uno", "unos", "usted", "ya", "y",
    "yo", "video", "youtube"
}


def fold(text: str) -> str:
    """Version sin tildes para comparar nombres, no para reemplazar IDs."""
    return "".join(
        c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c)
    ).casefold().strip()


def split_hashtag(match: re.Match[str]) -> str:
    tag = match.group(1).replace("_", " ")
    tag = re.sub(r"(?<=[a-záéíóúñ])(?=[A-ZÁÉÍÓÚÑ])", " ", tag)
    return " " + tag + " "


def clean_text(value: object) -> str:
    text = str(value)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text, flags=re.I)
    text = re.sub(r"#([\wÁÉÍÓÚÜÑáéíóúüñ]+)", split_hashtag, text)
    text = re.sub(r"@[\w.\-]+", " ", text)
    text = text.casefold()
    # Conserva letras y tildes; elimina cifras, puntuacion y emojis.
    text = re.sub(r"[^a-záéíóúüñ\s]", " ", text)
    tokens = [t for t in text.split() if len(t) > 1 and fold(t) not in STOPWORDS]
    return " ".join(tokens)


def parse_count(value: object, blank_is_zero: bool = False) -> float:
    """Convierte conteos con comas, puntos y sufijos K/M a numero."""
    text = str(value).strip().lower().replace("vistas", "").strip()
    if not text:
        return 0.0 if blank_is_zero else np.nan
    text = text.replace("\u00a0", "").replace(" ", "")
    multiplier = 1.0
    if text.endswith("k"):
        multiplier, text = 1_000.0, text[:-1]
    elif text.endswith("m"):
        multiplier, text = 1_000_000.0, text[:-1]
    if multiplier == 1.0:
        text = text.replace(",", "").replace(".", "")
    else:
        text = text.replace(",", ".")
    try:
        return float(text) * multiplier
    except ValueError:
        return np.nan


def shorten(text: str, length: int = 52) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text if len(text) <= length else text[: length - 1].rstrip() + "…"


def savefig(name: str) -> None:
    plt.tight_layout()
    plt.savefig(FIGURES / name, bbox_inches="tight", facecolor="white")
    plt.close()

## 1. Carga, comprensión e integración

Cada fila de `youtube_videos.csv` representa un video y su llave es `video_id`. Cada fila de `youtube_comments.csv` representa un comentario principal y su llave es `comment_id`. La integración se valida como muchos-a-uno mediante `video_id`.

In [ ]:
videos = pd.read_csv(DATA / "youtube_videos.csv", keep_default_na=False)
comments = pd.read_csv(DATA / "youtube_comments.csv", keep_default_na=False)

# Copias de auditoria antes de cualquier transformacion.
videos_raw = videos.copy()
comments_raw = comments.copy()

# Normalizacion conservadora: espacios externos en identificadores y campos visibles.
for col in ["video_id", "channel_id", "comment_id", "author_channel_id"]:
    target = videos if col in videos.columns else comments
    target[col] = target[col].astype(str).str.strip()
for col in ["title", "channel_name", "channel_handle", "owner_handle"]:
    videos[col] = videos[col].astype(str).str.strip()
for col in ["video_title", "channel_name", "author_name", "author_handle"]:
    comments[col] = comments[col].astype(str).str.strip()

videos["view_count_parsed"] = videos["view_count_text"].map(parse_count)
comments["like_count"] = comments["like_count_text"].map(
    lambda x: parse_count(x, blank_is_zero=True)
).astype("Int64")
comments["reply_count"] = pd.to_numeric(comments["reply_count"], errors="coerce").astype("Int64")
videos["publish_date"] = pd.to_datetime(videos["publish_date"], errors="coerce", utc=True)
videos["upload_date"] = pd.to_datetime(videos["upload_date"], errors="coerce", utc=True)

comments["texto_original"] = comments["text"].astype(str)
comments["texto_limpio"] = comments["texto_original"].map(clean_text)

# Integracion many-to-one: conserva un comentario por fila.
video_lookup_cols = [
    "video_id", "title", "channel_name", "channel_id", "category", "view_count",
    "publish_date", "channel_handle"
]
integrated = comments.merge(
    videos[video_lookup_cols], on="video_id", how="left", validate="many_to_one",
    indicator=True, suffixes=("_comentario", "_video")
)

In [ ]:
display(pd.DataFrame({
    'conjunto': ['videos', 'comentarios', 'integrado'],
    'filas': [len(videos_raw), len(comments_raw), len(integrated)],
    'columnas': [videos_raw.shape[1], comments_raw.shape[1], integrated.shape[1]]
}))
display(integrated['_merge'].value_counts().rename_axis('resultado').reset_index(name='comentarios'))

## 2. Calidad, limpieza y preprocesamiento

Se revisan dimensiones, vacíos, duplicados, constantes, valores atípicos y consistencia entre identificadores y etiquetas. Los atípicos se diagnostican, pero no se eliminan porque son plausibles en conteos de plataforma.

In [ ]:
def quality_rows(df: pd.DataFrame, dataset: str, primary_key: str) -> list[dict]:
    return [
        {"conjunto": dataset, "indicador": "filas", "valor": len(df)},
        {"conjunto": dataset, "indicador": "columnas", "valor": df.shape[1]},
        {"conjunto": dataset, "indicador": "faltantes/celdas vacias", "valor": int(df.isna().sum().sum() + (df.astype(str).apply(lambda s: s.str.strip().eq("")).sum().sum()))},
        {"conjunto": dataset, "indicador": "filas duplicadas", "valor": int(df.duplicated().sum())},
        {"conjunto": dataset, "indicador": f"{primary_key} duplicada", "valor": int(df[primary_key].duplicated().sum())},
        {"conjunto": dataset, "indicador": "variables constantes", "valor": int((df.nunique(dropna=False) <= 1).sum())},
    ]


quality = pd.DataFrame(
    quality_rows(videos_raw, "videos", "video_id")
    + quality_rows(comments_raw, "comentarios", "comment_id")
)
quality.to_csv(TABLES / "diagnostico_calidad.csv", index=False, encoding="utf-8-sig")

missing = []
for dataset, df in [("videos", videos_raw), ("comentarios", comments_raw)]:
    for col in df.columns:
        n = int(df[col].isna().sum() + df[col].astype(str).str.strip().eq("").sum())
        if n:
            missing.append({"conjunto": dataset, "variable": col, "faltantes": n, "porcentaje": 100 * n / len(df)})
pd.DataFrame(missing).to_csv(TABLES / "valores_faltantes.csv", index=False, encoding="utf-8-sig")

constants = []
for dataset, df in [("videos", videos_raw), ("comentarios", comments_raw)]:
    for col in df.columns:
        if df[col].nunique(dropna=False) <= 1:
            constants.append({"conjunto": dataset, "variable": col, "valor": str(df[col].iloc[0])})
pd.DataFrame(constants).to_csv(TABLES / "variables_constantes.csv", index=False, encoding="utf-8-sig")


def iqr_outliers(series: pd.Series) -> dict:
    s = pd.to_numeric(series, errors="coerce").dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {
        "q1": float(q1), "mediana": float(s.median()), "q3": float(q3),
        "limite_superior": float(upper), "atipicos_superiores": int((s > upper).sum()),
        "maximo": float(s.max())
    }


outlier_table = pd.DataFrame(
    [
        {"variable": "view_count", **iqr_outliers(videos["view_count"])},
        {"variable": "like_count", **iqr_outliers(comments["like_count"])},
        {"variable": "reply_count", **iqr_outliers(comments["reply_count"])},
    ]
)
outlier_table.to_csv(TABLES / "diagnostico_atipicos.csv", index=False, encoding="utf-8-sig")

# Consistencia entre llaves y campos redundantes.
matched = integrated["_merge"].eq("both")
channel_id_mismatch = matched & integrated["channel_id_comentario"].ne(integrated["channel_id_video"])
channel_name_mismatch = matched & integrated["channel_name_comentario"].map(fold).ne(integrated["channel_name_video"].map(fold))
title_mismatch = matched & integrated["video_title"].map(fold).ne(integrated["title"].map(fold))


def ambiguous_mapping(df: pd.DataFrame, key: str, value: str) -> int:
    return int((df.groupby(key)[value].nunique() > 1).sum())


consistency = pd.DataFrame(
    [
        {"comprobacion": "comentarios asociados a video", "valor": int(matched.sum())},
        {"comprobacion": "comentarios sin video", "valor": int((~matched).sum())},
        {"comprobacion": "diferencias channel_id al integrar", "valor": int(channel_id_mismatch.sum())},
        {"comprobacion": "diferencias channel_name al integrar", "valor": int(channel_name_mismatch.sum())},
        {"comprobacion": "diferencias de titulo al integrar", "valor": int(title_mismatch.sum())},
        {"comprobacion": "channel_id con mas de un nombre (videos)", "valor": ambiguous_mapping(videos, "channel_id", "channel_name")},
        {"comprobacion": "channel_id con mas de un handle (videos)", "valor": ambiguous_mapping(videos, "channel_id", "channel_handle")},
        {"comprobacion": "author_channel_id con mas de un nombre", "valor": ambiguous_mapping(comments, "author_channel_id", "author_name")},
        {"comprobacion": "author_channel_id con mas de un handle", "valor": ambiguous_mapping(comments, "author_channel_id", "author_handle")},
        {"comprobacion": "publish_date distinta de upload_date", "valor": int(videos["publish_date"].ne(videos["upload_date"]).sum())},
        {"comprobacion": "owner_handle distinto de channel_handle", "valor": int(videos["owner_handle"].ne(videos["channel_handle"]).sum())},
    ]
)
consistency.to_csv(TABLES / "consistencia.csv", index=False, encoding="utf-8-sig")

view_valid = videos["view_count_parsed"].notna()
view_mismatch = view_valid & videos["view_count_parsed"].ne(videos["view_count"])

In [ ]:
display(quality)
display(pd.DataFrame(missing))
display(pd.DataFrame(constants))
display(outlier_table)
display(consistency)

### Efecto de la limpieza textual

`texto_original` conserva el comentario para auditoría. `texto_limpio` elimina URL, menciones, puntuación, cifras, emojis y palabras vacías; los hashtags se separan y se conservan como términos. No se aplica lematización sin un modelo contextual validado para español guatemalteco.

In [ ]:
# Cuantificacion de limpieza de texto.
text_effect = {
    "registros": len(comments),
    "modificados": int(comments["texto_original"].ne(comments["texto_limpio"]).sum()),
    "vacios_antes": int(comments["texto_original"].str.strip().eq("").sum()),
    "vacios_despues": int(comments["texto_limpio"].str.strip().eq("").sum()),
    "duplicados_antes": int(comments["texto_original"].duplicated().sum()),
    "duplicados_despues": int(comments["texto_limpio"].duplicated().sum()),
    "registros_eliminados": 0,
}
pd.DataFrame([text_effect]).to_csv(TABLES / "efecto_limpieza_texto.csv", index=False, encoding="utf-8-sig")

In [ ]:
display(pd.DataFrame([text_effect]))
display(comments[['texto_original', 'texto_limpio']].head(10))

## 3. Análisis exploratorio

Se describen videos, canales, comentarios, autores, categorías, consultas, visualizaciones, respuestas, me gusta, palabras, bigramas y hashtags.

In [ ]:
# Resumen exploratorio.
n_videos = videos["video_id"].nunique()
n_channels = videos["channel_id"].nunique()
n_comments = comments["comment_id"].nunique()
n_authors = comments["author_channel_id"].nunique()
n_commented_videos = comments["video_id"].nunique()

overview = pd.DataFrame(
    {
        "elemento": ["Videos", "Canales", "Comentarios", "Autores", "Videos con comentarios"],
        "cantidad": [n_videos, n_channels, n_comments, n_authors, n_commented_videos],
    }
)
overview.to_csv(TABLES / "resumen_general.csv", index=False, encoding="utf-8-sig")

videos_by_channel = (
    videos.groupby(["channel_id", "channel_name"], as_index=False)
    .agg(videos=("video_id", "nunique"), visualizaciones=("view_count", "sum"))
    .sort_values(["videos", "visualizaciones"], ascending=False)
)
videos_by_channel.to_csv(TABLES / "videos_por_canal.csv", index=False, encoding="utf-8-sig")

per_video = (
    comments.groupby("video_id", as_index=False)
    .agg(comentarios=("comment_id", "nunique"), autores=("author_channel_id", "nunique"), respuestas=("reply_count", "sum"), me_gusta=("like_count", "sum"))
    .merge(videos[["video_id", "title", "channel_name", "channel_id", "view_count", "category"]], on="video_id", how="right")
    .fillna({"comentarios": 0, "autores": 0, "respuestas": 0, "me_gusta": 0})
)
for col in ["comentarios", "autores", "respuestas", "me_gusta"]:
    per_video[col] = per_video[col].astype(int)
per_video.sort_values(["comentarios", "view_count"], ascending=False).to_csv(
    TABLES / "resumen_por_video.csv", index=False, encoding="utf-8-sig"
)

channel_participation = (
    per_video.groupby(["channel_id", "channel_name"], as_index=False)
    .agg(videos=("video_id", "nunique"), comentarios=("comentarios", "sum"), autores=("autores", "sum"), visualizaciones=("view_count", "sum"))
    .sort_values(["comentarios", "visualizaciones"], ascending=False)
)
# Autores unicos a nivel canal (evita sumar duplicados entre videos).
channel_unique_authors = comments.groupby("channel_id")["author_channel_id"].nunique()
channel_participation["autores_unicos"] = channel_participation["channel_id"].map(channel_unique_authors).fillna(0).astype(int)
channel_participation.to_csv(TABLES / "resumen_por_canal.csv", index=False, encoding="utf-8-sig")

category_summary = (
    per_video.groupby("category", as_index=False)
    .agg(videos=("video_id", "nunique"), visualizaciones=("view_count", "sum"), comentarios=("comentarios", "sum"))
    .sort_values("videos", ascending=False)
)
category_summary.to_csv(TABLES / "resumen_categorias.csv", index=False, encoding="utf-8-sig")

query_summary = (
    videos.groupby(["source_group", "source_query"], as_index=False)
    .agg(videos=("video_id", "nunique"), visualizaciones=("view_count", "sum"))
    .sort_values("videos", ascending=False)
)
query_summary.to_csv(TABLES / "resumen_consultas.csv", index=False, encoding="utf-8-sig")

all_clean_tokens = [token for text in comments["texto_limpio"] for token in text.split()]
word_counts = Counter(all_clean_tokens)
bigram_counts = Counter(
    pair for text in comments["texto_limpio"] for pair in zip(text.split(), text.split()[1:])
)

hashtag_counter = Counter()
for series in [videos["title"], videos["description"], comments["texto_original"]]:
    for text in series:
        hashtag_counter.update(tag.casefold() for tag in re.findall(r"#([\wÁÉÍÓÚÜÑáéíóúüñ]+)", str(text)))

pd.DataFrame(word_counts.most_common(30), columns=["palabra", "frecuencia"]).to_csv(
    TABLES / "palabras_frecuentes.csv", index=False, encoding="utf-8-sig"
)
pd.DataFrame(
    [(" ".join(k), v) for k, v in bigram_counts.most_common(30)],
    columns=["bigrama", "frecuencia"]
).to_csv(TABLES / "bigramas_frecuentes.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(hashtag_counter.most_common(30), columns=["hashtag", "frecuencia"]).to_csv(
    TABLES / "hashtags_frecuentes.csv", index=False, encoding="utf-8-sig"
)

In [ ]:
display(overview)
display(per_video.sort_values(['comentarios', 'view_count'], ascending=False).head(10))
display(channel_participation.head(10))
display(category_summary)
display(query_summary.head(15))

### Concentración, visibilidad y audiencias compartidas

Las coincidencias de autores entre videos son un diagnóstico exploratorio. No sustituyen las proyecciones formales del ejercicio 5.

In [ ]:
# Concentracion.
video_comment_counts = per_video.sort_values("comentarios", ascending=False)["comentarios"].to_numpy()
channel_comment_counts = channel_participation.sort_values("comentarios", ascending=False)["comentarios"].to_numpy()


def top_share(values: np.ndarray, n: int) -> float:
    return 100 * values[:n].sum() / values.sum() if values.sum() else 0.0


concentration = pd.DataFrame(
    [
        {"nivel": "video", "grupo": "top 1", "porcentaje_comentarios": top_share(video_comment_counts, 1)},
        {"nivel": "video", "grupo": "top 5", "porcentaje_comentarios": top_share(video_comment_counts, 5)},
        {"nivel": "video", "grupo": "top 10", "porcentaje_comentarios": top_share(video_comment_counts, 10)},
        {"nivel": "canal", "grupo": "top 1", "porcentaje_comentarios": top_share(channel_comment_counts, 1)},
        {"nivel": "canal", "grupo": "top 5", "porcentaje_comentarios": top_share(channel_comment_counts, 5)},
        {"nivel": "canal", "grupo": "top 10", "porcentaje_comentarios": top_share(channel_comment_counts, 10)},
    ]
)
concentration.to_csv(TABLES / "concentracion_comentarios.csv", index=False, encoding="utf-8-sig")

# Relacion entre popularidad y participacion. Spearman evita asumir linealidad.
corr_pearson = float(per_video["view_count"].corr(per_video["comentarios"], method="pearson"))
corr_spearman = float(per_video["view_count"].corr(per_video["comentarios"], method="spearman"))
corr_spearman_commented = float(
    per_video.loc[per_video["comentarios"] > 0, "view_count"].corr(
        per_video.loc[per_video["comentarios"] > 0, "comentarios"], method="spearman"
    )
)

# Audiencias compartidas y autores recurrentes (diagnostico descriptivo; no es la proyeccion formal del ejercicio 5).
author_video_degree = comments.groupby("author_channel_id")["video_id"].nunique().sort_values(ascending=False)
recurring_authors = author_video_degree[author_video_degree > 1]
author_names = comments.drop_duplicates("author_channel_id").set_index("author_channel_id")["author_name"]
recurring_table = recurring_authors.rename("videos_comentados").reset_index()
recurring_table["autor"] = recurring_table["author_channel_id"].map(author_names)
recurring_table.to_csv(TABLES / "autores_recurrentes.csv", index=False, encoding="utf-8-sig")

video_authors = comments.groupby("video_id")["author_channel_id"].apply(set).to_dict()
shared_pairs = []
for left, right in combinations(sorted(video_authors), 2):
    shared = len(video_authors[left] & video_authors[right])
    if shared:
        shared_pairs.append((left, right, shared))
shared_pairs.sort(key=lambda row: row[2], reverse=True)
titles = videos.set_index("video_id")["title"]
shared_df = pd.DataFrame(shared_pairs, columns=["video_id_1", "video_id_2", "autores_compartidos"])
if not shared_df.empty:
    shared_df["video_1"] = shared_df["video_id_1"].map(titles)
    shared_df["video_2"] = shared_df["video_id_2"].map(titles)
shared_df.to_csv(TABLES / "audiencias_compartidas.csv", index=False, encoding="utf-8-sig")

In [ ]:
display(concentration)
display(recurring_table)
display(shared_df[['video_1', 'video_2', 'autores_compartidos']] if not shared_df.empty else shared_df)

## 4. Construcción de la red bipartita autor-video

Una arista indica que un autor publicó al menos un comentario principal en un video. Su peso es el número de comentarios de ese par. No representa amistad, respuesta directa, conversación ni aprobación.

In [ ]:
# Red bipartita completa: 293 videos, incluidos los que no tienen comentarios.
B = nx.Graph()
for row in videos.itertuples(index=False):
    B.add_node(
        f"v:{row.video_id}", tipo="video", video_id=row.video_id,
        etiqueta=row.title, canal=row.channel_name, categoria=row.category,
        visualizaciones=int(row.view_count)
    )
author_meta = comments.drop_duplicates("author_channel_id")
for row in author_meta.itertuples(index=False):
    B.add_node(
        f"a:{row.author_channel_id}", tipo="autor", author_channel_id=row.author_channel_id,
        etiqueta=row.author_name, handle=row.author_handle
    )
edge_counts = (
    comments.groupby(["author_channel_id", "video_id"], as_index=False)
    .agg(peso=("comment_id", "nunique"))
)
for row in edge_counts.itertuples(index=False):
    B.add_edge(f"a:{row.author_channel_id}", f"v:{row.video_id}", peso=int(row.peso))

node_rows = []
for node, attrs in B.nodes(data=True):
    node_rows.append(
        {
            "node_id": node,
            "tipo": attrs["tipo"],
            "etiqueta": attrs.get("etiqueta", ""),
            "grado": B.degree(node),
            "grado_ponderado": int(B.degree(node, weight="peso")),
            "canal": attrs.get("canal", ""),
            "categoria": attrs.get("categoria", ""),
            "visualizaciones": attrs.get("visualizaciones", ""),
            "handle": attrs.get("handle", ""),
        }
    )
pd.DataFrame(node_rows).to_csv(TABLES / "nodos_bipartita.csv", index=False, encoding="utf-8-sig")

edge_table = edge_counts.copy()
edge_table["source"] = "a:" + edge_table["author_channel_id"]
edge_table["target"] = "v:" + edge_table["video_id"]
edge_table["autor"] = edge_table["author_channel_id"].map(author_names)
edge_table["video"] = edge_table["video_id"].map(titles)
edge_table[["source", "target", "peso", "author_channel_id", "video_id", "autor", "video"]].to_csv(
    TABLES / "aristas_bipartita.csv", index=False, encoding="utf-8-sig"
)

In [ ]:
display(pd.DataFrame(node_rows).head(10))
display(edge_table[['source', 'target', 'peso', 'autor', 'video']].head(10))
print(f'Nodos: {B.number_of_nodes():,} | Aristas: {B.number_of_edges():,} | Suma de pesos: {sum(d["peso"] for _, _, d in B.edges(data=True)):,}')

## Visualizaciones del avance

In [ ]:
# Figuras.
topv = per_video.nlargest(10, "comentarios").sort_values("comentarios")
plt.figure(figsize=(7.0, 4.3))
plt.barh([shorten(x, 48) for x in topv["title"]], topv["comentarios"], color="#3f78a8")
plt.xlabel("Comentarios observados")
plt.title("Videos con mayor participacion")
for i, value in enumerate(topv["comentarios"]):
    plt.text(value + 0.4, i, str(value), va="center", fontsize=8)
savefig("01_top_videos_comentarios.png")

topc = channel_participation.nlargest(10, "comentarios").sort_values("comentarios")
plt.figure(figsize=(7.0, 4.0))
plt.barh([shorten(x, 35) for x in topc["channel_name"]], topc["comentarios"], color="#7b4f9d")
plt.xlabel("Comentarios observados")
plt.title("Canales con mayor participacion")
for i, value in enumerate(topc["comentarios"]):
    plt.text(value + 0.4, i, str(value), va="center", fontsize=8)
savefig("02_top_canales_comentarios.png")

sorted_counts = np.sort(video_comment_counts)[::-1]
cum = np.cumsum(sorted_counts) / sorted_counts.sum() * 100
plt.figure(figsize=(6.8, 3.7))
plt.plot(np.arange(1, len(cum) + 1), cum, color="#1f5f3b", linewidth=1.8)
plt.axhline(80, color="black", linestyle="--", linewidth=0.8)
plt.xlabel("Numero de videos, ordenados por comentarios")
plt.ylabel("Porcentaje acumulado de comentarios")
plt.title("Concentracion de comentarios por video")
savefig("03_concentracion_videos.png")

plt.figure(figsize=(6.5, 4.2))
plt.scatter(per_video["view_count"] + 1, per_video["comentarios"] + 0.15,
            alpha=0.65, s=24, color="#b05252", edgecolor="none")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Visualizaciones + 1 (escala logaritmica)")
plt.ylabel("Comentarios + 0.15 (escala logaritmica)")
plt.title("Visualizaciones y comentarios por video")
plt.text(0.02, 0.96, f"Spearman = {corr_spearman:.3f}", transform=plt.gca().transAxes, va="top")
savefig("04_vistas_comentarios.png")

fig, axes = plt.subplots(1, 2, figsize=(8.0, 4.0))
words_top = pd.DataFrame(word_counts.most_common(12), columns=["item", "n"]).sort_values("n")
bigrams_top = pd.DataFrame(
    [(" ".join(k), v) for k, v in bigram_counts.most_common(12)], columns=["item", "n"]
).sort_values("n")
axes[0].barh(words_top["item"], words_top["n"], color="#2f6f8f")
axes[0].set_title("Palabras")
axes[0].set_xlabel("Frecuencia")
axes[1].barh(bigrams_top["item"], bigrams_top["n"], color="#ba7b2c")
axes[1].set_title("Bigramas")
axes[1].set_xlabel("Frecuencia")
fig.suptitle("Terminos frecuentes en comentarios limpios", y=1.02)
savefig("05_palabras_bigramas.png")

hashtags_top = pd.DataFrame(hashtag_counter.most_common(12), columns=["hashtag", "n"]).sort_values("n")
plt.figure(figsize=(6.8, 3.6))
if len(hashtags_top):
    plt.barh(hashtags_top["hashtag"], hashtags_top["n"], color="#367a62")
else:
    plt.text(0.5, 0.5, "No se encontraron hashtags", ha="center", va="center")
plt.xlabel("Frecuencia")
plt.title("Hashtags mas frecuentes en videos y comentarios")
savefig("06_hashtags.png")

In [ ]:
from IPython.display import Image, display
for archivo in [
    '01_top_videos_comentarios.png',
    '02_top_canales_comentarios.png',
    '03_concentracion_videos.png',
    '04_vistas_comentarios.png',
    '05_palabras_bigramas.png',
    '06_hashtags.png',
]:
    display(Image(filename=str(FIGURES / archivo), width=850))

### Red completa

Se conservan los 293 videos, incluidos aquellos sin comentarios observados, para mostrar la cobertura real de los archivos entregados.

In [ ]:
# Visualizacion completa: posiciones por componentes para mostrar tambien los aislados.
components = sorted(nx.connected_components(B), key=len, reverse=True)
pos = {}
main = B.subgraph(components[0]) if components else B
if len(main):
    main_pos = nx.spring_layout(main, seed=19, k=max(0.12, 2.2 / np.sqrt(len(main))), iterations=130, weight="peso")
    for node, (x, y) in main_pos.items():
        pos[node] = (x * 3.8, y * 3.8)
small_components = components[1:]
cols = 16
for i, comp in enumerate(small_components):
    center = (5.4 + (i % cols) * 0.34, 3.2 - (i // cols) * 0.34)
    if len(comp) == 1:
        pos[next(iter(comp))] = center
    else:
        local = nx.circular_layout(B.subgraph(comp), scale=0.12, center=center)
        pos.update(local)

plt.figure(figsize=(10.5, 7.5))
video_nodes = [n for n, d in B.nodes(data=True) if d["tipo"] == "video"]
author_nodes = [n for n, d in B.nodes(data=True) if d["tipo"] == "autor"]
nx.draw_networkx_edges(B, pos, width=[0.18 + 0.22 * B[u][v]["peso"] for u, v in B.edges()], alpha=0.24, edge_color="#777777")
nx.draw_networkx_nodes(B, pos, nodelist=author_nodes, node_size=9, node_color="#3676a3", alpha=0.80, linewidths=0)
nx.draw_networkx_nodes(B, pos, nodelist=video_nodes, node_size=18, node_color="#c05b43", alpha=0.86, linewidths=0, node_shape="s")
plt.scatter([], [], s=22, color="#3676a3", label="Autor")
plt.scatter([], [], s=30, color="#c05b43", marker="s", label="Video")
plt.legend(frameon=False, loc="lower left")
plt.title("Red bipartita autor-video completa")
plt.axis("off")
savefig("07_red_bipartita_completa.png")

In [ ]:
display(Image(filename=str(FIGURES / '07_red_bipartita_completa.png'), width=1000))

## Exportación de resultados

In [ ]:
# Metricas que alimentan el informe.
nonisolated_components = [c for c in components if len(c) > 1]
articulation = list(nx.articulation_points(B)) if B.number_of_edges() else []
articulation_authors = [n for n in articulation if n.startswith("a:")]
articulation_videos = [n for n in articulation if n.startswith("v:")]

metrics = {
    "videos": n_videos,
    "canales": n_channels,
    "comentarios": n_comments,
    "autores": n_authors,
    "videos_con_comentarios": n_commented_videos,
    "comentarios_integrados": int(matched.sum()),
    "comentarios_sin_video": int((~matched).sum()),
    "view_text_validos": int(view_valid.sum()),
    "view_text_diferencias_con_view_count": int(view_mismatch.sum()),
    "likes_blancos_convertidos_cero": int(comments_raw["like_count_text"].astype(str).str.strip().eq("").sum()),
    "correlacion_pearson_vistas_comentarios": corr_pearson,
    "correlacion_spearman_vistas_comentarios": corr_spearman,
    "correlacion_spearman_solo_comentados": corr_spearman_commented,
    "autores_recurrentes": int(len(recurring_authors)),
    "max_videos_por_autor": int(author_video_degree.max()),
    "pares_video_con_audiencia_compartida": int(len(shared_df)),
    "nodos_bipartita": B.number_of_nodes(),
    "nodos_autor": len(author_nodes),
    "nodos_video": len(video_nodes),
    "aristas_bipartita": B.number_of_edges(),
    "suma_pesos_aristas": int(sum(d["peso"] for _, _, d in B.edges(data=True))),
    "videos_aislados": int(sum(B.degree(n) == 0 for n in video_nodes)),
    "componentes_totales": nx.number_connected_components(B),
    "componentes_no_triviales": len(nonisolated_components),
    "tamano_componente_mayor": len(components[0]) if components else 0,
    "autores_articulacion_preliminar": len(articulation_authors),
    "videos_articulacion_preliminar": len(articulation_videos),
    "texto": text_effect,
    "top_palabras": word_counts.most_common(10),
    "top_bigramas": [(" ".join(k), v) for k, v in bigram_counts.most_common(10)],
    "top_hashtags": hashtag_counter.most_common(10),
}
with (TABLES / "metricas.json").open("w", encoding="utf-8") as fh:
    json.dump(metrics, fh, ensure_ascii=False, indent=2)

# Datos limpios para auditoria y continuacion del laboratorio.
videos.to_csv(DATA / "youtube_videos_limpio.csv", index=False, encoding="utf-8-sig")
comments.to_csv(DATA / "youtube_comments_limpio.csv", index=False, encoding="utf-8-sig")
integrated.drop(columns="_merge").to_csv(DATA / "youtube_integrado.csv", index=False, encoding="utf-8-sig")

print(json.dumps(metrics, ensure_ascii=False, indent=2))

## Alcance

Este avance termina en el ejercicio 4. Las proyecciones, topología, comunidades, centralidades formales y sentimiento corresponden a la entrega final.